In [1]:
from context_shortening.get_ontology_descriptions import get_ai_taxonomy

ai_terms = get_ai_taxonomy("label")
ai_texts = get_ai_taxonomy("description")

/home/sondre/miniconda3/envs/deepseek_llmdap_v2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embedding_model_id = "all-MiniLM-L6-v2"



from sentence_transformers import SentenceTransformer, util
import torch

import pandas as pd

embedding_model = SentenceTransformer(embedding_model_id, device="cuda:1" if torch.cuda.device_count()>1 else "cuda:0")
ai_text_embedding = embedding_model.encode(ai_terms)

In [1]:
correct_categories = [
    "topic",
    "subject",
    "field",
    "subfield",
    "process",
    "method",
    "structure",
    "algorithm",]

wrong_categoires = [
    "company",
    "corporation",
    "product",
    "software product",
    "trademark",
    "brand",
    "journal",
    "person",
    "individual",
    "award",
    "competition",
]

In [2]:
correct_categories = [
    *correct_categories,
    *["AI related "+cat for cat in correct_categories],
    *["ML related "+cat for cat in correct_categories],
]
wrong_categoires = [
    *wrong_categoires,
    *["AI related "+cat for cat in wrong_categoires],
    *["ML related "+cat for cat in wrong_categoires],
]

In [3]:
wrong_categoires

['company',
 'corporation',
 'product',
 'software product',
 'trademark',
 'brand',
 'journal',
 'person',
 'individual',
 'award',
 'competition',
 'AI related company',
 'AI related corporation',
 'AI related product',
 'AI related software product',
 'AI related trademark',
 'AI related brand',
 'AI related journal',
 'AI related person',
 'AI related individual',
 'AI related award',
 'AI related competition',
 'ML related company',
 'ML related corporation',
 'ML related product',
 'ML related software product',
 'ML related trademark',
 'ML related brand',
 'ML related journal',
 'ML related person',
 'ML related individual',
 'ML related award',
 'ML related competition']

In [5]:
#correct_embedding = embedding_model.encode(correct_categories)
#wrong_embedding = embedding_model.encode(wrong_categoires)
category_embedding = embedding_model.encode([*correct_categories, *wrong_categoires])
correct_threshold = len(correct_categories)

In [6]:
similarities = util.cos_sim(ai_text_embedding, category_embedding)


In [7]:
import torch
k = 10 # try out differnt values

def get_topk_indices_per_row(tensor: torch.Tensor, k: int = 5) -> torch.Tensor:
    """
    Returns the column indices of the top-k values in each row of the input tensor.

    Args:
        tensor (torch.Tensor): A 2D tensor of shape (num_rows, num_columns).
        k (int): Number of top values to retrieve per row.

    Returns:
        torch.Tensor: A 2D tensor of shape (num_rows, k) containing the column indices.
    """
    _, indices = torch.topk(tensor, k=k, dim=1)
    return indices



In [8]:
topk = get_topk_indices_per_row(similarities, k=3)

In [9]:
n_correct = (topk < correct_threshold).sum(axis=1)

In [10]:
n_correct

tensor([3, 3, 3,  ..., 2, 3, 3])

In [11]:
df = pd.DataFrame(ai_terms, columns=["label"])
df["number_of_correct_in_topk"] = n_correct

In [12]:
df = df

In [13]:
df[df["number_of_correct_in_topk"]==3].sample(frac=1)

,label,number_of_correct_in_topk
1633,Multispectral pattern recognition,3
4066,Bees algorithm,3
3930,Recursive neural network,3
425,Cognitive philology,3
1081,Multifactor dimensionality reduction,3
...,...,...
448,Concurrent MetateM,3
1502,Event condition action,3
1356,Page Analysis and Ground Truth Elements,3
3866,Instantaneously trained neural networks,3


In [14]:
df[df["number_of_correct_in_topk"]==2].sample(frac=1)

,label,number_of_correct_in_topk
589,Open-source artificial intelligence,2
5323,Opponent process,2
5387,Blend modes,2
801,Guideline execution engine,2
3842,Traffic Electronic Control System (Turkey),2
...,...,...
3533,Lusser's law,2
2291,Regulation of algorithms,2
2607,Convolutional neural network,2
2109,Voice stress analysis,2


In [15]:
df[df["number_of_correct_in_topk"]==1].sample(frac=1)

,label,number_of_correct_in_topk
2729,Possibility theory,1
2472,Water remote sensing,1
5590,Social inertia,1
6806,Perfect spline,1
556,Australian Artificial Intelligence Institute,1
...,...,...
6908,WinBUGS,1
4641,TCEC Season 20,1
1699,Prescription monitoring program,1
5317,Resolution enhancement technology,1


In [16]:
df[df["number_of_correct_in_topk"]==0].sample(frac=1)

,label,number_of_correct_in_topk
5008,Terabot-S,0
1402,Microsoft Azure Form Recognizer,0
153,PropBank,0
1526,Digital organism,0
1222,Keras,0
...,...,...
6554,Walking vehicle,0
498,Artificial intelligence industry in China,0
5033,Regulation of unmanned aerial vehicles,0
6508,Omnibot,0


In [17]:
df["description"] = ai_texts

In [18]:
df

,label,number_of_correct_in_topk,description
0,Natural language processing,3,Natural language processing (NLP) is a subfiel...
1,Artificial intelligence,3,Artificial intelligence (AI) is intelligence—p...
2,Machine translation,3,"Machine translation, sometimes referred to by ..."
3,Knowledge representation and reasoning,3,"Knowledge representation and reasoning (KRR, K..."
4,Computational linguistics,3,Computational linguistics is an interdisciplin...
...,...,...,...
7485,Disorder problem,3,In the study of stochastic processes in mathem...
7486,Scenario optimization,2,The scenario approach or scenario optimization...
7487,Wald's maximin model,2,"In decision theory and game theory, Wald's max..."
7488,Automatic basis function construction,3,"In machine learning, automatic basis function ..."


In [19]:
# dont want these
df.loc[df["label"].str.contains("List of"), "number_of_correct_in_topk"] = 0
df.loc[df["label"].str.contains("History of"), "number_of_correct_in_topk"] = 0

In [20]:
df_relevant = df[df["number_of_correct_in_topk"]==3]


In [21]:
df_relevant.to_csv("dbpedia_aitax_restricted_top3.csv")